# 1. Image Preprocessing for inference(pytorch)

# write a function to load an image and preprocess it for (PyTorch)

In [2]:
from PIL import Image
from torchvision import transforms

def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([ transforms.Resize((224, 224)),
                                     transforms.ToTensor(),
                                     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
                                   ])
    return transform(image).unsqueeze(0)

In [ ]:
! pip install torchvision

# 2. Predict on New Image with a Trained Model Perform prediction and get the class label

In [ ]:
model.eval()
input_image = preprocess_image("test.jpg")
with torch.no_grad():
    output = model(input_image)
    predicted_class = output.argmax(1).item()
print("Predicted_class:", predicted_class)

# 3. Build a CNN to classify CIFAR-10 image (PyTorch)

# Create a CNN model that classifies images from the CIFAR-10 dataset with accuracy above 60%

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

#preprocessing

transform = transforms.Compose([ transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,))
                               ])

trainset = torchvision.datasets.CIFAR10(root = './data', train = True, download = True, transform = transform)
testset = torchvision.datasets.CIFAR10(root = './data', train = False, download = True, transform = transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = torch.utils.data.DataLoader(testset, batch_size = 64, shuffle = True)

#CNN model:

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv = nn.Sequential( nn.Conv2d(3, 32, 3, padding=1),
                                  nn.ReLU(),
                                  nn.MaxPool2d(2),
                                  nn.Conv2d(32, 64, 3, padding=1),
                                  nn.ReLU(),
                                  nn.MaxPool2d(2))
        self.fc = nn.Sequential( nn.Linear(64*8*8, 128),
                                nn.ReLU(),
                                nn.Linear(128, 10))
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr = 0.001)

# training (1 epoch shown for simplicity)

for epoch in range(1):
    for images, labels in trainloader:
        outputs = net(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        
                    
                                          

100%|██████████| 170M/170M [31:54<00:00, 89.0kB/s]   


# Question 4: Identify Overfitting from Training Logs and Solve It
# Problem: You notice the training accuracy increases but validation accuracy stagnates. Modify the model using dropout and early stopping(use mnist dataset)

In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train/255.0, x_test/255.0
y_train, y_test = to_categorical(y_train), to_categorical(y_test)

model = Sequential([
    Flatten (input_shape=(28, 28)),
    Dense(256, activation = 'relu'),
    Dropout(0.3),
    Dense(128, activation = 'relu'),
    Dropout(0.3),
    Dense(10, activation ='softmax')])

model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
early_stop = EarlyStopping(patience =3, restore_best_weights = True)

model.fit(x_train, y_train, validation_data = (x_test, y_test), epochs = 30, callbacks = [early_stop])

C:\Anaconda3\envs\aiml\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9093 - loss: 0.3036 - val_accuracy: 0.9626 - val_loss: 0.1243
Epoch 2/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9561 - loss: 0.1465 - val_accuracy: 0.9723 - val_loss: 0.0874
Epoch 3/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9653 - loss: 0.1154 - val_accuracy: 0.9736 - val_loss: 0.0860
Epoch 4/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9700 - loss: 0.0984 - val_accuracy: 0.9750 - val_loss: 0.0823
Epoch 5/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9725 - loss: 0.0865 - val_accuracy: 0.9782 - val_loss: 0.0744
Epoch 6/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9765 - loss: 0.0769 - val_accuracy: 0.9792 - val_loss: 0.0759
Epoch 7/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9780 - loss: 0.0713 - val_accuracy: 0.9796 - val_loss: 0.0678
Epoch 8/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9797 - loss: 0.0661 -

## Question 5: Transfer Learning with Pretrained VGG16 (Cats vs Dogs)
# Problem: Use VGG16 for binary classification with fine-tuning

In [6]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout

base_model = VGG16(include_top = False, input_shape = (224, 224, 3), weights ='imagenet')
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = Flatten()(x)
x = Dropout(0.5)(x)
x = Dense (128, activation = 'relu') (x)
output = Dense(1, activation = 'sigmoid')(x)

model = Model(inputs=base_model.input, outputs = output)
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
